# Reproduction notebook: 22_itransformer_plus_frozen_historical_memory

This notebook is retained as an executable provenance record for the anonymous supplementary package. Saved outputs and internal development notes have been removed.


In [ ]:

from pathlib import Path
from types import SimpleNamespace
from contextlib import nullcontext

import gc
import importlib
import math
import random
import subprocess
import sys
import time
import warnings

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 240)
pd.set_option("display.width", 500)

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

DATASETS = [
    "ETTm1",
    "ETTh1",
    "Weather",
    "Electricity",
]

HORIZONS = [
    96,
    192,
    336,
    720,
]

TASKS = [
    (dataset, horizon)
    for dataset in DATASETS
    for horizon in HORIZONS
]

DIRECT_SEQ_LEN = 96
LABEL_LEN = 48
RET_SEQ_LEN = 96

DIRECT_SEED = 2023
CROSSFIT_SEED = 2222

TOP_K = 10
MEMORY_STRIDE = 24
OOF_ANCHOR_STRIDE = 4

FOLDS = [
    (0.55, 0.70),
    (0.70, 0.85),
    (0.85, 1.00),
]

# Frozen predictive representation.
REP_PATCH_LEN = 16
REP_PATCH_STRIDE = 16
REP_D_MODEL = 64
REP_N_HEADS = 4
REP_LAYERS = 2
REP_D_FF = 128
REP_DIM = 64
REP_DROPOUT = 0.1

REP_NUM_PATCHES = (
    1
    + (
        RET_SEQ_LEN
        - REP_PATCH_LEN
    )
    // REP_PATCH_STRIDE
)

# Frozen adaptive-gate design.
GATE_DIM = 26
GATE_LR = 1e-3
GATE_WD = 1e-4
GATE_BATCH = 8192
GATE_MAX_EPOCHS = 50
GATE_PATIENCE = 7

ALPHA_GRID = np.round(
    np.arange(
        0.0,
        1.0001,
        0.1,
    ),
    10,
)

LAMBDA_GRID = np.array(
    [
        0.0,
        0.25,
        0.50,
        0.75,
        1.00,
    ],
    dtype=np.float32,
)

EPS = 1e-8
RETRIEVER_USE_AMP = torch.cuda.is_available()

# Small retrieval batches prevent GPU blow-up on 321-channel Electricity.
TARGET_RETRIEVAL_PAIRS = 672

# Direct iTransformer is evaluated in much larger anchor blocks.
DIRECT_ANCHOR_BLOCK = {
    "ETTm1": 128,
    "ETTh1": 128,
    "Weather": 64,
    "Electricity": 16,
}

EXP21_ROOT = Path(
    "/data/dataset/strong_forecaster/"
    "itransformer_official_baseline_reproduction"
)

FROZEN_RET_ROOT = Path(
    "/data/dataset/strong_forecaster/"
    "multidataset_crossfit_screening"
)

ROOT = Path(
    "/data/dataset/strong_forecaster/"
    "itransformer_plus_frozen_historical_memory"
)

DIRS = {
    "fold_direct": ROOT / "fold_direct",
    "oof": ROOT / "oof",
    "validation": ROOT / "validation",
    "memory_emb": ROOT / "memory_embeddings",
    "gate": ROOT / "gate",
    "history": ROOT / "history",
    "paired": ROOT / "paired_test",
    "channel": ROOT / "channel_test",
    "calibration": ROOT / "calibration",
    "artifacts": ROOT / "artifacts",
}

for p in DIRS.values():
    p.mkdir(
        parents=True,
        exist_ok=True,
    )

RESUME = True
FORCE = False

DIRECT_PARITY_TOL = 2e-5

print("Device:", DEVICE)
print("Tasks:", len(TASKS))
print("Experiment 21:", EXP21_ROOT)
print("Frozen retrievers:", FROZEN_RET_ROOT)
print("Output:", ROOT)


In [ ]:

REPO_CANDIDATES = [
    Path(
        "/code/stock_regime_retrieval/"
        "strong_forecaster/iTransformer_official"
    ),
    Path("/code/iTransformer"),
    Path("/data/iTransformer"),
]

REPO = next(
    (
        p
        for p in REPO_CANDIDATES
        if (
            p / "model" / "iTransformer.py"
        ).is_file()
    ),
    None,
)

if REPO is None:
    raise FileNotFoundError(
        "Official iTransformer repository was not found. "
        "Experiment 21 must be available before Experiment 22."
    )

for module_name in list(
    sys.modules.keys()
):
    if (
        module_name == "model"
        or module_name.startswith("model.")
        or module_name == "layers"
        or module_name.startswith("layers.")
        or module_name == "utils"
        or module_name.startswith("utils.")
    ):
        del sys.modules[
            module_name
        ]

if str(REPO) in sys.path:
    sys.path.remove(
        str(REPO)
    )

sys.path.insert(
    0,
    str(REPO),
)

itransformer_module = importlib.import_module(
    "model.iTransformer"
)

tools_module = importlib.import_module(
    "utils.tools"
)

timefeatures_module = importlib.import_module(
    "utils.timefeatures"
)

OfficialITransformer = (
    itransformer_module.Model
)

official_adjust_lr = (
    tools_module.adjust_learning_rate
)

official_time_features = (
    timefeatures_module.time_features
)

actual_model_file = Path(
    itransformer_module.__file__
).resolve()

expected_model_file = (
    REPO
    / "model"
    / "iTransformer.py"
).resolve()

if (
    actual_model_file
    != expected_model_file
):
    raise RuntimeError(
        "Wrong iTransformer implementation imported.\n"
        f"Expected: {expected_model_file}\n"
        f"Actual:   {actual_model_file}"
    )

try:
    commit = subprocess.check_output(
        [
            "git",
            "-C",
            str(REPO),
            "rev-parse",
            "HEAD",
        ],
        text=True,
    ).strip()
except Exception:
    commit = "unknown"

print(
    "PASS: official iTransformer implementation."
)
print("Repo:", REPO)
print("Commit:", commit)

(
    DIRS["artifacts"]
    / "official_repo_commit.txt"
).write_text(
    commit + "\n"
)


In [ ]:

EXP21_SUMMARY_PATH = (
    EXP21_ROOT
    / "summary.csv"
)

if not EXP21_SUMMARY_PATH.is_file():
    raise FileNotFoundError(
        f"Experiment 21 summary not found: "
        f"{EXP21_SUMMARY_PATH}"
    )

EXP21_SUMMARY = pd.read_csv(
    EXP21_SUMMARY_PATH
)

display(
    EXP21_SUMMARY[
        [
            "Dataset",
            "Horizon",
            "BestEpoch",
            "Test_MSE",
            "Test_MAE",
        ]
    ].sort_values(
        [
            "Dataset",
            "Horizon",
        ]
    )
)


RECIPES = {
    "ETTm1": {
        "data": "ETTm1",
        "enc_in": 7,
        "e_layers": 2,
        "d_model": 512,
        "d_ff": 2048,
        "n_heads": 8,
        "dropout": 0.1,
        "batch_size": 32,
        "learning_rate": 1e-4,
        "train_epochs": 10,
        "patience": 3,
        "lradj": "type1",
        "factor": 1,
        "freq": "t",
    },
    "ETTh1": {
        "data": "ETTh1",
        "enc_in": 7,
        "e_layers": 2,
        "d_model": 512,
        "d_ff": 2048,
        "n_heads": 8,
        "dropout": 0.1,
        "batch_size": 32,
        "learning_rate": 1e-4,
        "train_epochs": 10,
        "patience": 3,
        "lradj": "type1",
        "factor": 1,
        "freq": "h",
    },
    "Weather": {
        "data": "custom",
        "enc_in": 21,
        "e_layers": 3,
        "d_model": 512,
        "d_ff": 512,
        "n_heads": 8,
        "dropout": 0.1,
        "batch_size": 32,
        "learning_rate": 1e-4,
        "train_epochs": 10,
        "patience": 3,
        "lradj": "type1",
        "factor": 1,
        "freq": "h",
    },
    "Electricity": {
        "data": "custom",
        "enc_in": 321,
        "e_layers": 3,
        "d_model": 512,
        "d_ff": 512,
        "n_heads": 8,
        "dropout": 0.1,
        "batch_size": 16,
        "learning_rate": 5e-4,
        "train_epochs": 10,
        "patience": 3,
        "lradj": "type1",
        "factor": 1,
        "freq": "h",
    },
}


def itransformer_config(
    name,
    horizon,
):
    r = RECIPES[
        name
    ]

    return SimpleNamespace(
        task_name=
            "long_term_forecast",
        seq_len=
            DIRECT_SEQ_LEN,
        label_len=
            LABEL_LEN,
        pred_len=
            int(
                horizon
            ),
        output_attention=
            False,
        enc_in=
            r[
                "enc_in"
            ],
        dec_in=
            r[
                "enc_in"
            ],
        c_out=
            r[
                "enc_in"
            ],
        d_model=
            r[
                "d_model"
            ],
        embed=
            "timeF",
        freq=
            r[
                "freq"
            ],
        dropout=
            r[
                "dropout"
            ],
        factor=
            r[
                "factor"
            ],
        n_heads=
            r[
                "n_heads"
            ],
        d_ff=
            r[
                "d_ff"
            ],
        activation=
            "gelu",
        e_layers=
            r[
                "e_layers"
            ],
        d_layers=
            1,
        class_strategy=
            "projection",
        use_norm=
            1,
    )


def build_itransformer(
    name,
    horizon,
):
    return OfficialITransformer(
        itransformer_config(
            name,
            horizon,
        )
    ).float().to(
        DEVICE
    )


def exp21_ckpt_path(
    name,
    horizon,
):
    return (
        EXP21_ROOT
        / "checkpoints"
        / (
            f"{name}_H{horizon}_"
            f"seed{DIRECT_SEED}.pt"
        )
    )


def exp21_reference(
    name,
    horizon,
):
    hit = EXP21_SUMMARY[
        (
            EXP21_SUMMARY[
                "Dataset"
            ]
            == name
        )
        & (
            EXP21_SUMMARY[
                "Horizon"
            ]
            == horizon
        )
    ]

    if len(
        hit
    ) != 1:
        raise RuntimeError(
            f"Expected one Experiment 21 row "
            f"for {name} H={horizon}."
        )

    return hit.iloc[
        0
    ]


In [ ]:

DATA_PATH_CANDIDATES = {
    "ETTm1": [
        Path("/data/dataset/ETTm1.csv"),
        Path(
            "/data/Time-Series-Library/"
            "dataset/ETT-small/ETTm1.csv"
        ),
        Path(
            "/data/Time-Series-Library_v2/"
            "dataset/ETT-small/ETTm1.csv"
        ),
    ],
    "ETTh1": [
        Path("/data/dataset/ETTh1.csv"),
        Path(
            "/data/Time-Series-Library/"
            "dataset/ETT-small/ETTh1.csv"
        ),
        Path(
            "/data/Time-Series-Library_v2/"
            "dataset/ETT-small/ETTh1.csv"
        ),
    ],
    "Weather": [
        Path("/data/dataset/weather.csv"),
        Path("/data/dataset/weather/weather.csv"),
        Path(
            "/data/Time-Series-Library/"
            "dataset/weather/weather.csv"
        ),
        Path(
            "/data/Time-Series-Library_v2/"
            "dataset/weather/weather.csv"
        ),
    ],
    "Electricity": [
        Path("/data/dataset/electricity.csv"),
        Path(
            "/data/dataset/electricity/"
            "electricity.csv"
        ),
        Path(
            "/data/Time-Series-Library/"
            "dataset/electricity/electricity.csv"
        ),
        Path(
            "/data/Time-Series-Library_v2/"
            "dataset/electricity/electricity.csv"
        ),
    ],
}

DATA_PATHS = {}

for name, candidates in (
    DATA_PATH_CANDIDATES.items()
):
    hit = next(
        (
            p
            for p in candidates
            if p.is_file()
        ),
        None,
    )

    DATA_PATHS[
        name
    ] = hit

    print(
        f"{name:11s}:",
        hit if hit is not None else "NOT FOUND",
    )

if any(
    p is None
    for p in DATA_PATHS.values()
):
    raise FileNotFoundError(
        "At least one required dataset CSV is missing."
    )


def standard_boundaries(
    name,
    n,
):
    if name == "ETTh1":
        train_end = (
            12
            * 30
            * 24
        )

        val_end = (
            (
                12
                + 4
            )
            * 30
            * 24
        )

        test_end = (
            (
                12
                + 8
            )
            * 30
            * 24
        )

    elif name == "ETTm1":
        unit = (
            30
            * 24
            * 4
        )

        train_end = (
            12
            * unit
        )

        val_end = (
            (
                12
                + 4
            )
            * unit
        )

        test_end = (
            (
                12
                + 8
            )
            * unit
        )

    else:
        train_end = int(
            n
            * 0.7
        )

        num_test = int(
            n
            * 0.2
        )

        num_val = (
            n
            - train_end
            - num_test
        )

        val_end = (
            train_end
            + num_val
        )

        test_end = n

    if test_end > n:
        raise ValueError(
            f"{name}: dataset is shorter than "
            f"the official split. "
            f"Need {test_end}, found {n}."
        )

    return (
        train_end,
        val_end,
        test_end,
    )


def prepare_data(
    name,
):
    path = DATA_PATHS[
        name
    ]

    df = pd.read_csv(
        path
    )

    if "date" not in df.columns:
        raise ValueError(
            f"{name}: no date column."
        )

    if (
        name
        in [
            "Weather",
            "Electricity",
        ]
    ):
        if "OT" not in df.columns:
            raise ValueError(
                f"{name}: no OT column."
            )

        cols = list(
            df.columns
        )

        cols.remove(
            "OT"
        )

        cols.remove(
            "date"
        )

        df = df[
            ["date"]
            + cols
            + ["OT"]
        ].copy()

    value_cols = list(
        df.columns[
            1:
        ]
    )

    raw = df[
        value_cols
    ].to_numpy(
        dtype=np.float64
    )

    (
        train_end,
        val_end,
        test_end,
    ) = standard_boundaries(
        name,
        len(
            raw
        ),
    )

    scaler = StandardScaler()

    scaler.fit(
        raw[
            :train_end
        ]
    )

    z = scaler.transform(
        raw
    ).astype(
        np.float32
    )

    dates = pd.to_datetime(
        df[
            "date"
        ].values
    )

    marks = official_time_features(
        dates,
        freq=
            RECIPES[
                name
            ][
                "freq"
            ],
    ).transpose(
        1,
        0,
    ).astype(
        np.float32
    )

    C = raw.shape[
        1
    ]

    if (
        C
        != RECIPES[
            name
        ][
            "enc_in"
        ]
    ):
        raise ValueError(
            f"{name}: expected "
            f"{RECIPES[name]['enc_in']} channels, "
            f"found {C}."
        )

    return {
        "name":
            name,
        "path":
            path,
        "columns":
            value_cols,
        "raw":
            raw,
        "z":
            z,
        "marks":
            marks,
        "n_channels":
            C,
        "train_end":
            train_end,
        "val_end":
            val_end,
        "test_end":
            test_end,
        "full_scaler":
            scaler,
    }


DATA = {
    name:
        prepare_data(
            name
        )
    for name in DATASETS
}

display(
    pd.DataFrame([
        {
            "Dataset":
                name,
            "RowsUsed":
                d[
                    "test_end"
                ],
            "Channels":
                d[
                    "n_channels"
                ],
            "TrainEnd":
                d[
                    "train_end"
                ],
            "ValEnd":
                d[
                    "val_end"
                ],
            "TestEnd":
                d[
                    "test_end"
                ],
            "TimeFeatures":
                d[
                    "marks"
                ].shape[
                    1
                ],
        }
        for name, d
        in DATA.items()
    ])
)


In [ ]:

def prefix_normalize(
    raw,
    prefix,
):
    scaler = StandardScaler()

    scaler.fit(
        raw[
            :prefix
        ]
    )

    z = scaler.transform(
        raw
    ).astype(
        np.float32
    )

    return (
        z,
        scaler,
    )


In [ ]:

def set_seed(
    seed,
):
    random.seed(
        seed
    )

    np.random.seed(
        seed
    )

    torch.manual_seed(
        seed
    )

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(
            seed
        )

    torch.backends.cudnn.benchmark = False


def load_torch(
    path,
):
    try:
        return torch.load(
            path,
            map_location=DEVICE,
            weights_only=False,
        )
    except TypeError:
        return torch.load(
            path,
            map_location=DEVICE,
        )


def load_exp21_direct(
    name,
    horizon,
):
    path = exp21_ckpt_path(
        name,
        horizon,
    )

    ckpt = load_torch(
        path
    )

    model = build_itransformer(
        name,
        horizon,
    )

    model.load_state_dict(
        ckpt[
            "StateDict"
        ]
    )

    model.eval()

    return (
        model,
        ckpt,
    )


def eval_anchors(
    start,
    end,
    horizon,
    stride=1,
):
    return np.arange(
        max(
            int(
                start
            ),
            DIRECT_SEQ_LEN,
        ),
        int(
            end
        )
        - int(
            horizon
        )
        + 1,
        int(
            stride
        ),
        dtype=np.int64,
    )


def make_direct_batch(
    z,
    marks,
    anchors,
    horizon,
):
    anchors = np.asarray(
        anchors,
        dtype=np.int64,
    )

    x_idx = (
        anchors[
            :,
            None
        ]
        - DIRECT_SEQ_LEN
        + np.arange(
            DIRECT_SEQ_LEN
        )[
            None,
            :
        ]
    )

    y_idx = (
        anchors[
            :,
            None
        ]
        - LABEL_LEN
        + np.arange(
            LABEL_LEN
            + horizon
        )[
            None,
            :
        ]
    )

    x = z[
        x_idx,
        :
    ].astype(
        np.float32
    )

    y = z[
        y_idx,
        :
    ].astype(
        np.float32
    )

    x_mark = marks[
        x_idx,
        :
    ].astype(
        np.float32
    )

    y_mark = marks[
        y_idx,
        :
    ].astype(
        np.float32
    )

    return (
        x,
        y,
        x_mark,
        y_mark,
    )


def forward_direct(
    model,
    x,
    y,
    x_mark,
    y_mark,
    horizon,
):
    x_t = torch.from_numpy(
        x
    ).to(
        DEVICE
    )

    y_t = torch.from_numpy(
        y
    ).to(
        DEVICE
    )

    xm_t = torch.from_numpy(
        x_mark
    ).to(
        DEVICE
    )

    ym_t = torch.from_numpy(
        y_mark
    ).to(
        DEVICE
    )

    dec_inp = torch.zeros_like(
        y_t[
            :,
            -horizon:,
            :
        ]
    )

    dec_inp = torch.cat(
        [
            y_t[
                :,
                :LABEL_LEN,
                :
            ],
            dec_inp,
        ],
        dim=1,
    )

    pred = model(
        x_t,
        xm_t,
        dec_inp,
        ym_t,
    )

    pred = pred[
        :,
        -horizon:,
        :
    ].float()

    true = y_t[
        :,
        -horizon:,
        :
    ].float()

    return (
        pred,
        true,
        x_t,
        y_t,
        xm_t,
        ym_t,
        dec_inp,
    )


@torch.no_grad()
def direct_residual_block(
    model,
    z,
    marks,
    anchors,
    horizon,
):
    (
        x,
        y,
        x_mark,
        y_mark,
    ) = make_direct_batch(
        z,
        marks,
        anchors,
        horizon,
    )

    (
        pred,
        true,
        x_t,
        y_t,
        xm_t,
        ym_t,
        dec_inp,
    ) = forward_direct(
        model,
        x,
        y,
        x_mark,
        y_mark,
        horizon,
    )

    current = x_t[
        :,
        -1:,
        :
    ].float()

    pred_residual = (
        pred
        - current
    )

    true_residual = (
        true
        - current
    )

    del (
        true,
        x_t,
        y_t,
        xm_t,
        ym_t,
        dec_inp,
    )

    return (
        pred_residual,
        true_residual,
    )


@torch.no_grad()
def evaluate_direct_parity(
    name,
    horizon,
    model,
):
    d = DATA[
        name
    ]

    anchors = eval_anchors(
        d[
            "val_end"
        ],
        d[
            "test_end"
        ],
        horizon,
        stride=1,
    )

    sse = 0.0
    sae = 0.0
    n = 0

    block = DIRECT_ANCHOR_BLOCK[
        name
    ]

    for i in range(
        0,
        len(
            anchors
        ),
        block,
    ):
        a = anchors[
            i:
            i+block
        ]

        pred_r, true_r = (
            direct_residual_block(
                model,
                d[
                    "z"
                ],
                d[
                    "marks"
                ],
                a,
                horizon,
            )
        )

        e = (
            pred_r
            - true_r
        )

        sse += float(
            (
                e
                * e
            ).sum()
        )

        sae += float(
            e.abs().sum()
        )

        n += e.numel()

        del (
            pred_r,
            true_r,
            e,
        )

    return (
        sse
        / n,
        sae
        / n,
        len(
            anchors
        ),
    )


In [ ]:

PARITY_PATH = (
    DIRS[
        "artifacts"
    ]
    / "direct_parity.csv"
)

parity_rows = []

for name, horizon in TASKS:
    model, ckpt = load_exp21_direct(
        name,
        horizon,
    )

    mse, mae, windows = (
        evaluate_direct_parity(
            name,
            horizon,
            model,
        )
    )

    ref = exp21_reference(
        name,
        horizon,
    )

    mse_diff = abs(
        mse
        - float(
            ref[
                "Test_MSE"
            ]
        )
    )

    mae_diff = abs(
        mae
        - float(
            ref[
                "Test_MAE"
            ]
        )
    )

    parity_rows.append({
        "Dataset":
            name,
        "Horizon":
            horizon,
        "Exp21_MSE":
            float(
                ref[
                    "Test_MSE"
                ]
            ),
        "Manual_MSE":
            mse,
        "MSEAbsDiff":
            mse_diff,
        "Exp21_MAE":
            float(
                ref[
                    "Test_MAE"
                ]
            ),
        "Manual_MAE":
            mae,
        "MAEAbsDiff":
            mae_diff,
        "Windows":
            windows,
        "Pass":
            mse_diff
            < DIRECT_PARITY_TOL,
    })

    print(
        f"{name:11s} H={horizon:3d} | "
        f"Exp21={float(ref['Test_MSE']):.8f} | "
        f"manual={mse:.8f} | "
        f"diff={mse_diff:.2e} | "
        f"{'PASS' if mse_diff < DIRECT_PARITY_TOL else 'FAIL'}"
    )

    del (
        model,
        ckpt,
    )

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

parity_df = pd.DataFrame(
    parity_rows
)

parity_df.to_csv(
    PARITY_PATH,
    index=False,
)

display(
    parity_df
)

failed = parity_df[
    ~parity_df[
        "Pass"
    ]
]

if len(
    failed
):
    raise RuntimeError(
        "Direct parity failed. "
        "Do not start cross-fitting until this is fixed."
    )

print(
    "PASS: all 16 direct baselines match Experiment 21."
)


In [ ]:

def inv_softplus(
    x,
):
    return math.log(
        math.exp(
            float(
                x
            )
        )
        - 1.0
    )


class PredictivePatchEncoder(
    nn.Module
):
    def __init__(
        self,
    ):
        super().__init__()

        self.patch_proj = nn.Linear(
            REP_PATCH_LEN,
            REP_D_MODEL,
        )

        self.pos_embed = nn.Parameter(
            torch.zeros(
                1,
                REP_NUM_PATCHES,
                REP_D_MODEL,
            )
        )

        nn.init.trunc_normal_(
            self.pos_embed,
            std=0.02,
        )

        layer = nn.TransformerEncoderLayer(
            d_model=
                REP_D_MODEL,
            nhead=
                REP_N_HEADS,
            dim_feedforward=
                REP_D_FF,
            dropout=
                REP_DROPOUT,
            activation=
                "gelu",
            batch_first=
                True,
            norm_first=
                True,
        )

        self.encoder = nn.TransformerEncoder(
            layer,
            num_layers=
                REP_LAYERS,
        )

        self.norm = nn.LayerNorm(
            REP_D_MODEL
        )

        self.proj = nn.Linear(
            REP_D_MODEL,
            REP_DIM,
        )

    def forward(
        self,
        x,
    ):
        p = x.unfold(
            1,
            REP_PATCH_LEN,
            REP_PATCH_STRIDE,
        )

        h = (
            self.patch_proj(
                p
            )
            + self.pos_embed[
                :,
                :p.shape[
                    1
                ],
            ]
        )

        h = self.encoder(
            h
        ).mean(
            dim=1
        )

        h = self.proj(
            self.norm(
                h
            )
        )

        return F.normalize(
            h,
            dim=-1,
            eps=1e-8,
        )


class EmbeddingOnlyRetriever(
    nn.Module
):
    def __init__(
        self,
    ):
        super().__init__()

        self.encoder = (
            PredictivePatchEncoder()
        )

        self.raw_gamma = nn.Parameter(
            torch.tensor(
                inv_softplus(
                    1.0
                ),
                dtype=torch.float32,
            )
        )

    @property
    def gamma(
        self,
    ):
        return F.softplus(
            self.raw_gamma
        )

    def encode(
        self,
        x,
    ):
        return self.encoder(
            x
        )


def ret_amp():
    if RETRIEVER_USE_AMP:
        return torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
        )

    return nullcontext()


def full_retriever_ckpt_path(
    name,
    horizon,
):
    return (
        FROZEN_RET_ROOT
        / "full_retriever"
        / (
            f"{name}_H{horizon}_seed0.pt"
        )
    )


def fold_retriever_ckpt_path(
    name,
    horizon,
    fold,
):
    return (
        FROZEN_RET_ROOT
        / "fold_retriever"
        / (
            f"{name}_H{horizon}_"
            f"F{fold}_seed0.pt"
        )
    )


def load_frozen_retriever(
    path,
):
    ckpt = load_torch(
        path
    )

    model = EmbeddingOnlyRetriever().to(
        DEVICE
    )

    model.load_state_dict(
        ckpt[
            "StateDict"
        ]
    )

    model.eval()

    return (
        model,
        ckpt,
    )


In [ ]:

preflight_rows = []

for name, horizon in TASKS:
    preflight_rows.append({
        "Dataset":
            name,
        "Horizon":
            horizon,
        "Kind":
            "Exp21 direct",
        "Path":
            exp21_ckpt_path(
                name,
                horizon,
            ),
    })

    preflight_rows.append({
        "Dataset":
            name,
        "Horizon":
            horizon,
        "Kind":
            "Full retriever",
        "Path":
            full_retriever_ckpt_path(
                name,
                horizon,
            ),
    })

    for fold in range(
        1,
        4,
    ):
        preflight_rows.append({
            "Dataset":
                name,
            "Horizon":
                horizon,
            "Kind":
                f"Fold retriever F{fold}",
            "Path":
                fold_retriever_ckpt_path(
                    name,
                    horizon,
                    fold,
                ),
        })

preflight = pd.DataFrame(
    preflight_rows
)

preflight[
    "Exists"
] = preflight[
    "Path"
].map(
    lambda p:
        Path(
            p
        ).is_file()
)

display(
    preflight
)

missing = preflight[
    ~preflight[
        "Exists"
    ]
]

if len(
    missing
):
    for p in missing[
        "Path"
    ]:
        print(
            "MISSING:",
            p,
        )

    raise FileNotFoundError(
        "Required frozen checkpoint is missing."
    )

print(
    "PASS: all 80 prerequisite checkpoint references exist."
)


In [ ]:

def retrieval_anchor_batch(
    name,
):
    C = DATA[
        name
    ][
        "n_channels"
    ]

    return max(
        1,
        TARGET_RETRIEVAL_PAIRS
        // C,
    )


def batch_pattern(
    x,
):
    x = np.asarray(
        x,
        np.float32,
    )

    xc = (
        x
        - x.mean(
            axis=-1,
            keepdims=True,
        )
    )

    n = np.linalg.norm(
        xc,
        axis=-1,
        keepdims=True,
    )

    return np.where(
        n > EPS,
        xc
        / np.maximum(
            n,
            EPS,
        ),
        0.0,
    ).astype(
        np.float32
    )


def context7(
    x,
):
    x = np.asarray(
        x,
        np.float32,
    )

    short = max(
        8,
        RET_SEQ_LEN
        // 4,
    )

    m = x.mean(
        axis=-1
    )

    s = (
        x.std(
            axis=-1
        )
        + EPS
    )

    f1 = (
        x[
            ...,
            -1
        ]
        - m
    ) / s

    f2 = (
        x[
            ...,
            -short:
        ].mean(
            axis=-1
        )
        - m
    ) / s

    f3 = (
        x[
            ...,
            -1
        ]
        - x[
            ...,
            -short
        ]
    ) / s

    f4 = (
        x[
            ...,
            -1
        ]
        - x[
            ...,
            0
        ]
    ) / s

    df = np.diff(
        x,
        axis=-1,
    )

    ds = np.diff(
        x[
            ...,
            -short:
        ],
        axis=-1,
    )

    f5 = (
        ds.std(
            axis=-1
        )
        + EPS
    ) / (
        df.std(
            axis=-1
        )
        + EPS
    )

    t = np.linspace(
        -1.0,
        1.0,
        RET_SEQ_LEN,
        dtype=np.float32,
    )

    t = (
        t
        - t.mean()
    )

    f6 = (
        np.sum(
            t
            * (
                x
                - m[
                    ...,
                    None
                ]
            ),
            axis=-1,
        )
        / (
            np.sum(
                t
                * t
            )
            + EPS
        )
    ) / s

    a = x[
        ...,
        :-1
    ]

    b = x[
        ...,
        1:
    ]

    a = (
        a
        - a.mean(
            axis=-1,
            keepdims=True,
        )
    )

    b = (
        b
        - b.mean(
            axis=-1,
            keepdims=True,
        )
    )

    f7 = np.sum(
        a
        * b,
        axis=-1,
    ) / (
        np.sqrt(
            np.sum(
                a
                * a,
                axis=-1,
            )
            * np.sum(
                b
                * b,
                axis=-1,
            )
        )
        + EPS
    )

    return np.stack(
        [
            f1,
            f2,
            f3,
            f4,
            f5,
            f6,
            f7,
        ],
        axis=-1,
    ).astype(
        np.float32
    )


def extract_channel(
    z,
    c,
    anchors,
    horizon,
):
    anchors = np.asarray(
        anchors,
        np.int64,
    )

    pi = (
        anchors[
            :,
            None
        ]
        - RET_SEQ_LEN
        + np.arange(
            RET_SEQ_LEN
        )[
            None,
            :
        ]
    )

    fi = (
        anchors[
            :,
            None
        ]
        + np.arange(
            horizon
        )[
            None,
            :
        ]
    )

    past = z[
        pi,
        c,
    ].astype(
        np.float32
    )

    future = z[
        fi,
        c,
    ].astype(
        np.float32
    )

    current = z[
        anchors
        - 1,
        c,
    ].astype(
        np.float32
    )

    future_residual = (
        future
        - current[
            :,
            None
        ]
    ).astype(
        np.float32
    )

    return (
        past,
        future_residual,
    )


def build_memory(
    z,
    channels,
    boundary,
    horizon,
):
    memory_anchors = np.arange(
        RET_SEQ_LEN,
        int(
            boundary
        )
        - horizon
        + 1,
        MEMORY_STRIDE,
        dtype=np.int64,
    )

    if len(
        memory_anchors
    ) < TOP_K:
        raise ValueError(
            "Insufficient admissible memory."
        )

    past = np.empty(
        (
            channels,
            len(
                memory_anchors
            ),
            RET_SEQ_LEN,
        ),
        dtype=np.float32,
    )

    pattern = np.empty_like(
        past
    )

    future = np.empty(
        (
            channels,
            len(
                memory_anchors
            ),
            horizon,
        ),
        dtype=np.float32,
    )

    for c in range(
        channels
    ):
        p, f = extract_channel(
            z,
            c,
            memory_anchors,
            horizon,
        )

        past[
            c
        ] = p

        pattern[
            c
        ] = batch_pattern(
            p
        )

        future[
            c
        ] = f

    return {
        "anchors":
            memory_anchors,
        "past":
            past,
        "pattern":
            pattern,
        "future":
            future,
        "M":
            len(
                memory_anchors
            ),
        "boundary":
            int(
                boundary
            ),
    }


def query_pairs(
    z,
    anchors,
    channels,
    horizon,
):
    anchors = np.asarray(
        anchors,
        np.int64,
    )

    channels = np.asarray(
        channels,
        np.int64,
    )

    pi = (
        anchors[
            :,
            None
        ]
        - RET_SEQ_LEN
        + np.arange(
            RET_SEQ_LEN
        )[
            None,
            :
        ]
    )

    fi = (
        anchors[
            :,
            None
        ]
        + np.arange(
            horizon
        )[
            None,
            :
        ]
    )

    past = z[
        pi,
        channels[
            :,
            None
        ],
    ].astype(
        np.float32
    )

    future = z[
        fi,
        channels[
            :,
            None
        ],
    ].astype(
        np.float32
    )

    current = z[
        anchors
        - 1,
        channels,
    ].astype(
        np.float32
    )

    true_residual = (
        future
        - current[
            :,
            None
        ]
    ).astype(
        np.float32
    )

    return (
        past,
        batch_pattern(
            past
        ),
        context7(
            past
        ),
        true_residual,
    )


In [ ]:

@torch.no_grad()
def encode_np(
    model,
    x,
    chunk=512,
):
    parts = []

    for i in range(
        0,
        len(
            x
        ),
        chunk,
    ):
        t = torch.from_numpy(
            x[
                i:
                i+chunk
            ]
        ).to(
            DEVICE
        )

        with ret_amp():
            e = model.encode(
                t
            ).float()

        parts.append(
            e.cpu()
        )

        del (
            t,
            e,
        )

    return torch.cat(
        parts,
        dim=0,
    ).numpy().astype(
        np.float32
    )


def memory_embedding_path(
    name,
    horizon,
    tag,
):
    return (
        DIRS[
            "memory_emb"
        ]
        / (
            f"{name}_H{horizon}_"
            f"{tag}_emb.npy"
        )
    )


@torch.no_grad()
def memory_gpu_cached(
    name,
    horizon,
    tag,
    model,
    memory,
    channels,
):
    path = memory_embedding_path(
        name,
        horizon,
        tag,
    )

    expected = (
        channels,
        memory[
            "M"
        ],
        REP_DIM,
    )

    emb_np = None

    if (
        path.exists()
        and RESUME
        and not FORCE
    ):
        candidate = np.load(
            path,
            mmap_mode=None,
        )

        if (
            tuple(
                candidate.shape
            )
            == expected
        ):
            emb_np = candidate.astype(
                np.float32,
                copy=False,
            )

            print(
                "Loaded memory embedding:",
                path.name,
            )

    if emb_np is None:
        emb_np = np.empty(
            expected,
            dtype=np.float32,
        )

        print(
            "Building memory embedding:",
            path.name,
            expected,
        )

        for c in range(
            channels
        ):
            emb_np[
                c
            ] = encode_np(
                model,
                memory[
                    "past"
                ][
                    c
                ],
            )

            if (
                c == 0
                or (
                    c + 1
                )
                % 50
                == 0
                or (
                    c + 1
                    == channels
                )
            ):
                print(
                    f"  channel "
                    f"{c+1}/{channels}"
                )

        np.save(
            path,
            emb_np,
        )

    return {
        "emb":
            torch.from_numpy(
                emb_np
            ).to(
                DEVICE
            ),
        "pattern":
            torch.from_numpy(
                memory[
                    "pattern"
                ]
            ).to(
                DEVICE
            ),
        "future":
            torch.from_numpy(
                memory[
                    "future"
                ]
            ).to(
                DEVICE
            ),
    }


In [ ]:

@torch.no_grad()
def retrieve(
    model,
    memory_gpu_obj,
    z,
    anchors,
    channels,
    horizon,
):
    (
        past,
        pattern,
        ctx,
        true,
    ) = query_pairs(
        z,
        anchors,
        channels,
        horizon,
    )

    past_t = torch.from_numpy(
        past
    ).to(
        DEVICE
    )

    pattern_t = torch.from_numpy(
        pattern
    ).to(
        DEVICE
    )

    with ret_amp():
        qemb = model.encode(
            past_t
        )

    qemb = qemb.float()

    memb = memory_gpu_obj[
        "emb"
    ][
        channels
    ]

    sim = torch.bmm(
        qemb[
            :,
            None,
            :
        ],
        memb.transpose(
            1,
            2,
        ),
    ).squeeze(
        1
    )

    score = (
        model.gamma
        * sim
    )

    idx = torch.topk(
        score,
        TOP_K,
        dim=1,
    ).indices

    row = torch.arange(
        len(
            channels
        ),
        device=DEVICE,
    )[
        :,
        None
    ]

    pfull = torch.bmm(
        pattern_t[
            :,
            None,
            :
        ],
        memory_gpu_obj[
            "pattern"
        ][
            channels
        ].transpose(
            1,
            2,
        ),
    ).squeeze(
        1
    )

    return {
        "score":
            score[
                row,
                idx
            ],
        "sim":
            sim[
                row,
                idx
            ],
        "pattern":
            pfull[
                row,
                idx
            ],
        "cand":
            memory_gpu_obj[
                "future"
            ][
                channels[
                    :,
                    None
                ],
                idx,
            ],
        "ctx":
            torch.from_numpy(
                ctx
            ).to(
                DEVICE
            ),
        "true":
            torch.from_numpy(
                true
            ).to(
                DEVICE
            ),
    }


In [ ]:

class CrossFitAdaptiveGate(
    nn.Module
):
    def __init__(
        self,
    ):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(
                GATE_DIM,
                64,
            ),
            nn.LayerNorm(
                64
            ),
            nn.GELU(),
            nn.Dropout(
                0.1
            ),
            nn.Linear(
                64,
                32,
            ),
            nn.GELU(),
            nn.Dropout(
                0.1
            ),
            nn.Linear(
                32,
                1,
            ),
        )

        nn.init.normal_(
            self.net[
                -1
            ].weight,
            mean=0.0,
            std=1e-3,
        )

        nn.init.constant_(
            self.net[
                -1
            ].bias,
            math.log(
                0.1
                / 0.9
            ),
        )

    def forward(
        self,
        x,
    ):
        return torch.sigmoid(
            self.net(
                x
            ).squeeze(
                -1
            )
        )


def score_entropy(
    s,
):
    p = torch.softmax(
        s,
        dim=1,
    )

    return (
        -(
            p
            * torch.log(
                p.clamp_min(
                    1e-8
                )
            )
        ).sum(
            dim=1
        )
        / math.log(
            TOP_K
        )
    )


def feature_cosine(
    a,
    b,
):
    return (
        (
            a
            * b
        ).sum(
            dim=1
        )
        / (
            torch.sqrt(
                (
                    a
                    * a
                ).sum(
                    dim=1
                )
                + 1e-8
            )
            * torch.sqrt(
                (
                    b
                    * b
                ).sum(
                    dim=1
                )
                + 1e-8
            )
        )
    )


def gate_features(
    r,
    retrieval,
    direct,
):
    s = r[
        "score"
    ]

    sim = r[
        "sim"
    ]

    pattern = r[
        "pattern"
    ]

    sorted_s = torch.sort(
        s,
        dim=1,
        descending=True,
    ).values

    cand_std = r[
        "cand"
    ].std(
        dim=1,
        unbiased=False,
    )

    disp_rms = torch.sqrt(
        (
            cand_std
            * cand_std
        ).mean(
            dim=1
        )
        + 1e-8
    )

    disp_mean = cand_std.mean(
        dim=1
    )

    direct_rms = torch.sqrt(
        (
            direct
            * direct
        ).mean(
            dim=1
        )
        + 1e-8
    )

    retrieval_rms = torch.sqrt(
        (
            retrieval
            * retrieval
        ).mean(
            dim=1
        )
        + 1e-8
    )

    disagreement = (
        retrieval
        - direct
    )

    disagreement_rms = torch.sqrt(
        (
            disagreement
            * disagreement
        ).mean(
            dim=1
        )
        + 1e-8
    )

    relative_disagreement = (
        disagreement_rms
        / (
            direct_rms
            + retrieval_rms
            + 1e-6
        )
    )

    scalars = torch.stack(
        [
            s.mean(
                dim=1
            ),
            s.std(
                dim=1,
                unbiased=False,
            ),
            s.max(
                dim=1
            ).values,
            sorted_s[
                :,
                0
            ]
            - sorted_s[
                :,
                1
            ],
            s.max(
                dim=1
            ).values
            - s.mean(
                dim=1
            ),
            score_entropy(
                s
            ),
            sim.mean(
                dim=1
            ),
            sim.std(
                dim=1,
                unbiased=False,
            ),
            sim.max(
                dim=1
            ).values,
            pattern.mean(
                dim=1
            ),
            pattern.std(
                dim=1,
                unbiased=False,
            ),
            pattern.max(
                dim=1
            ).values,
            disp_rms,
            disp_mean,
            direct_rms,
            retrieval_rms,
            disagreement_rms,
            relative_disagreement,
            feature_cosine(
                direct,
                retrieval,
            ),
        ],
        dim=1,
    )

    out = torch.cat(
        [
            r[
                "ctx"
            ],
            scalars,
        ],
        dim=1,
    )

    if out.shape[
        1
    ] != GATE_DIM:
        raise RuntimeError(
            f"Gate feature dimension mismatch: "
            f"{out.shape}"
        )

    return out


def abc_terms(
    direct,
    retrieval,
    true,
):
    e = (
        direct
        - true
    )

    delta = (
        retrieval
        - direct
    )

    return torch.stack(
        [
            (
                e
                * e
            ).mean(
                dim=1
            ),
            (
                e
                * delta
            ).mean(
                dim=1
            ),
            (
                delta
                * delta
            ).mean(
                dim=1
            ),
        ],
        dim=1,
    )


In [ ]:

def fold_direct_path(
    name,
    horizon,
    fold,
):
    return (
        DIRS[
            "fold_direct"
        ]
        / (
            f"{name}_H{horizon}_"
            f"F{fold}_itransformer.pt"
        )
    )


def history_path(
    name,
    horizon,
    fold,
):
    return (
        DIRS[
            "history"
        ]
        / (
            f"{name}_H{horizon}_"
            f"F{fold}_direct_history.csv"
        )
    )


def train_anchors(
    prefix,
    horizon,
):
    return np.arange(
        DIRECT_SEQ_LEN,
        int(
            prefix
        )
        - horizon
        + 1,
        dtype=np.int64,
    )


def train_fold_itransformer(
    name,
    horizon,
    z,
    marks,
    prefix,
    fixed_epochs,
    fold,
):
    path = fold_direct_path(
        name,
        horizon,
        fold,
    )

    if (
        path.exists()
        and RESUME
        and not FORCE
    ):
        ckpt = load_torch(
            path
        )

        model = build_itransformer(
            name,
            horizon,
        )

        model.load_state_dict(
            ckpt[
                "StateDict"
            ]
        )

        model.eval()

        print(
            "Loaded fold direct:",
            path.name,
        )

        return (
            model,
            ckpt,
        )

    r = RECIPES[
        name
    ]

    seed = (
        CROSSFIT_SEED
        + horizon
        * 100
        + fold
        * 17
        + int(
            prefix
        )
        % 997
    )

    set_seed(
        seed
    )

    model = build_itransformer(
        name,
        horizon,
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=
            r[
                "learning_rate"
            ],
    )

    anchors = train_anchors(
        prefix,
        horizon,
    )

    rng = np.random.default_rng(
        seed
        + 1
    )

    history = []

    for epoch in range(
        1,
        int(
            fixed_epochs
        )
        + 1,
    ):
        model.train()

        order = rng.permutation(
            len(
                anchors
            )
        )

        usable = (
            len(
                order
            )
            // r[
                "batch_size"
            ]
        ) * r[
            "batch_size"
        ]

        order = order[
            :usable
        ]

        losses = []
        t0 = time.time()

        for left in range(
            0,
            usable,
            r[
                "batch_size"
            ],
        ):
            ids = order[
                left:
                left
                + r[
                    "batch_size"
                ]
            ]

            a = anchors[
                ids
            ]

            (
                x,
                y,
                x_mark,
                y_mark,
            ) = make_direct_batch(
                z,
                marks,
                a,
                horizon,
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            (
                pred,
                true,
                x_t,
                y_t,
                xm_t,
                ym_t,
                dec_inp,
            ) = forward_direct(
                model,
                x,
                y,
                x_mark,
                y_mark,
                horizon,
            )

            loss = F.mse_loss(
                pred,
                true,
            )

            loss.backward()
            optimizer.step()

            losses.append(
                float(
                    loss.item()
                )
            )

            del (
                pred,
                true,
                x_t,
                y_t,
                xm_t,
                ym_t,
                dec_inp,
                loss,
            )

        train_mse = float(
            np.mean(
                losses
            )
        )

        # Same type1 schedule as Experiment 21.
        args = SimpleNamespace(
            learning_rate=
                r[
                    "learning_rate"
                ],
            lradj=
                r[
                    "lradj"
                ],
            train_epochs=
                r[
                    "train_epochs"
                ],
        )

        official_adjust_lr(
            optimizer,
            epoch,
            args,
        )

        current_lr = optimizer.param_groups[
            0
        ][
            "lr"
        ]

        history.append({
            "Epoch":
                epoch,
            "TrainMSE":
                train_mse,
            "LR":
                current_lr,
            "Seconds":
                time.time()
                - t0,
        })

        pd.DataFrame(
            history
        ).to_csv(
            history_path(
                name,
                horizon,
                fold,
            ),
            index=False,
        )

        print(
            f"Fold direct {name:11s} "
            f"H={horizon:3d} F{fold} "
            f"ep={epoch:02d}/{fixed_epochs} "
            f"train={train_mse:.6f} "
            f"lr={current_lr:.3e}"
        )

    model.eval()

    state = {
        k:
            v.detach()
            .cpu()
            .clone()
        for k, v
        in model.state_dict().items()
    }

    ckpt = {
        "Dataset":
            name,
        "Horizon":
            horizon,
        "Fold":
            fold,
        "Prefix":
            int(
                prefix
            ),
        "FixedEpochs":
            int(
                fixed_epochs
            ),
        "Seed":
            seed,
        "StateDict":
            state,
    }

    torch.save(
        ckpt,
        path,
    )

    return (
        model,
        ckpt,
    )


In [ ]:

@torch.no_grad()
def collect_gate_data(
    data,
    horizon,
    direct_model,
    retriever,
    memory_gpu_obj,
    z,
    anchors,
):
    name = data[
        "name"
    ]

    C = data[
        "n_channels"
    ]

    ret_block = retrieval_anchor_batch(
        name
    )

    direct_block = DIRECT_ANCHOR_BLOCK[
        name
    ]

    features = []
    abcs = []
    anchors_out = []
    channels_out = []

    for outer in range(
        0,
        len(
            anchors
        ),
        direct_block,
    ):
        a_big = anchors[
            outer:
            outer+direct_block
        ]

        direct_big, true_big = (
            direct_residual_block(
                direct_model,
                z,
                data[
                    "marks"
                ],
                a_big,
                horizon,
            )
        )

        # [A, H, C] -> [A, C, H]
        direct_big = direct_big.permute(
            0,
            2,
            1,
        ).contiguous()

        true_big = true_big.permute(
            0,
            2,
            1,
        ).contiguous()

        for inner in range(
            0,
            len(
                a_big
            ),
            ret_block,
        ):
            a = a_big[
                inner:
                inner+ret_block
            ]

            A = len(
                a
            )

            pair_anchor = np.repeat(
                a,
                C,
            )

            pair_channel = np.tile(
                np.arange(
                    C,
                    dtype=np.int64,
                ),
                A,
            )

            r = retrieve(
                retriever,
                memory_gpu_obj,
                z,
                pair_anchor,
                pair_channel,
                horizon,
            )

            retrieval = r[
                "cand"
            ].mean(
                dim=1
            )

            d = direct_big[
                inner:
                inner+A
            ].reshape(
                -1,
                horizon,
            )

            t = true_big[
                inner:
                inner+A
            ].reshape(
                -1,
                horizon,
            )

            # Strong consistency check:
            # direct true residual must match retrieval true residual.
            max_true_diff = float(
                (
                    t
                    - r[
                        "true"
                    ]
                ).abs().max()
            )

            if (
                max_true_diff
                > 2e-5
            ):
                raise RuntimeError(
                    f"True residual mismatch: "
                    f"{max_true_diff}"
                )

            feat = gate_features(
                r,
                retrieval,
                d,
            )

            abc = abc_terms(
                d,
                retrieval,
                t,
            )

            features.append(
                feat.cpu()
                .numpy()
                .astype(
                    np.float32
                )
            )

            abcs.append(
                abc.cpu()
                .numpy()
                .astype(
                    np.float32
                )
            )

            anchors_out.append(
                pair_anchor
            )

            channels_out.append(
                pair_channel
            )

            del (
                r,
                retrieval,
                d,
                t,
                feat,
                abc,
            )

        del (
            direct_big,
            true_big,
        )

    return {
        "feature":
            np.concatenate(
                features,
                axis=0,
            ),
        "abc":
            np.concatenate(
                abcs,
                axis=0,
            ),
        "anchor":
            np.concatenate(
                anchors_out,
                axis=0,
            ),
        "channel":
            np.concatenate(
                channels_out,
                axis=0,
            ),
    }


In [ ]:

def oof_cache_path(
    name,
    horizon,
    fold,
):
    return (
        DIRS[
            "oof"
        ]
        / (
            f"{name}_H{horizon}_"
            f"F{fold}_oof.npz"
        )
    )


def val_cache_path(
    name,
    horizon,
):
    return (
        DIRS[
            "validation"
        ]
        / (
            f"{name}_H{horizon}_"
            "validation.npz"
        )
    )


def build_oof_fold(
    data,
    horizon,
    fold,
    p0,
    p1,
    fixed_epochs,
):
    name = data[
        "name"
    ]

    C = data[
        "n_channels"
    ]

    out_path = oof_cache_path(
        name,
        horizon,
        fold,
    )

    if (
        out_path.exists()
        and RESUME
        and not FORCE
    ):
        obj = np.load(
            out_path
        )

        print(
            "Loaded OOF cache:",
            out_path.name,
        )

        return {
            key:
                obj[
                    key
                ]
            for key in [
                "feature",
                "abc",
                "anchor",
                "channel",
            ]
        }

    prefix = int(
        p0
        * data[
            "train_end"
        ]
    )

    oof_end = int(
        p1
        * data[
            "train_end"
        ]
    )

    z, prefix_scaler = prefix_normalize(
        data[
            "raw"
        ],
        prefix,
    )

    direct_model, _ = (
        train_fold_itransformer(
            name,
            horizon,
            z,
            data[
                "marks"
            ],
            prefix,
            fixed_epochs,
            fold,
        )
    )

    retriever, _ = load_frozen_retriever(
        fold_retriever_ckpt_path(
            name,
            horizon,
            fold,
        )
    )

    memory = build_memory(
        z,
        C,
        prefix,
        horizon,
    )

    memory_gpu_obj = memory_gpu_cached(
        name,
        horizon,
        (
            f"F{fold}_"
            f"prefix{prefix}"
        ),
        retriever,
        memory,
        C,
    )

    anchors = eval_anchors(
        prefix,
        oof_end,
        horizon,
        stride=
            OOF_ANCHOR_STRIDE,
    )

    print(
        f"OOF {name} H={horizon} F{fold}: "
        f"prefix={prefix}, "
        f"end={oof_end}, "
        f"anchors={len(anchors)}, "
        f"pairs={len(anchors)*C}, "
        f"memory/C={memory['M']}"
    )

    out = collect_gate_data(
        data,
        horizon,
        direct_model,
        retriever,
        memory_gpu_obj,
        z,
        anchors,
    )

    np.savez_compressed(
        out_path,
        **out,
    )

    del (
        direct_model,
        retriever,
        memory,
        memory_gpu_obj,
        prefix_scaler,
        z,
    )

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return out


def build_validation_cache(
    data,
    horizon,
    direct_model,
    retriever,
):
    name = data[
        "name"
    ]

    C = data[
        "n_channels"
    ]

    path = val_cache_path(
        name,
        horizon,
    )

    if (
        path.exists()
        and RESUME
        and not FORCE
    ):
        obj = np.load(
            path
        )

        print(
            "Loaded validation cache:",
            path.name,
        )

        return {
            key:
                obj[
                    key
                ]
            for key in [
                "feature",
                "abc",
                "anchor",
                "channel",
            ]
        }

    memory = build_memory(
        data[
            "z"
        ],
        C,
        data[
            "train_end"
        ],
        horizon,
    )

    memory_gpu_obj = memory_gpu_cached(
        name,
        horizon,
        "validation_train_memory",
        retriever,
        memory,
        C,
    )

    anchors = eval_anchors(
        data[
            "train_end"
        ],
        data[
            "val_end"
        ],
        horizon,
        stride=1,
    )

    print(
        f"Validation {name} H={horizon}: "
        f"anchors={len(anchors)}, "
        f"pairs={len(anchors)*C}, "
        f"memory/C={memory['M']}"
    )

    out = collect_gate_data(
        data,
        horizon,
        direct_model,
        retriever,
        memory_gpu_obj,
        data[
            "z"
        ],
        anchors,
    )

    np.savez_compressed(
        path,
        **out,
    )

    del (
        memory,
        memory_gpu_obj,
    )

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return out


In [ ]:

def fit_feature_scaler(
    x,
):
    median = np.median(
        x,
        axis=0,
    ).astype(
        np.float32
    )

    q25 = np.percentile(
        x,
        25,
        axis=0,
    )

    q75 = np.percentile(
        x,
        75,
        axis=0,
    )

    iqr = (
        q75
        - q25
    ).astype(
        np.float32
    )

    iqr = np.where(
        iqr < 1e-5,
        1.0,
        iqr,
    ).astype(
        np.float32
    )

    return (
        median,
        iqr,
    )


def scale_features(
    x,
    median,
    iqr,
):
    return np.clip(
        (
            x
            - median
        )
        / iqr,
        -8.0,
        8.0,
    ).astype(
        np.float32
    )


def gate_loss(
    alpha,
    abc,
):
    return (
        abc[
            :,
            0
        ]
        + 2.0
        * alpha
        * abc[
            :,
            1
        ]
        + alpha
        * alpha
        * abc[
            :,
            2
        ]
    ).mean()


def gate_checkpoint_path(
    name,
    horizon,
):
    return (
        DIRS[
            "gate"
        ]
        / (
            f"{name}_H{horizon}_"
            "gate.pt"
        )
    )


def train_gate_epoch(
    model,
    optimizer,
    x,
    abc,
    rng,
):
    model.train()

    order = rng.permutation(
        len(
            x
        )
    )

    losses = []

    for i in range(
        0,
        len(
            order
        ),
        GATE_BATCH,
    ):
        ids = order[
            i:
            i+GATE_BATCH
        ]

        xt = torch.from_numpy(
            x[
                ids
            ]
        ).to(
            DEVICE
        )

        at = torch.from_numpy(
            abc[
                ids
            ]
        ).to(
            DEVICE
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        alpha = model(
            xt
        )

        loss = gate_loss(
            alpha,
            at,
        )

        loss.backward()
        optimizer.step()

        losses.append(
            float(
                loss.item()
            )
        )

        del (
            xt,
            at,
            alpha,
            loss,
        )

    return float(
        np.mean(
            losses
        )
    )


@torch.no_grad()
def evaluate_gate(
    model,
    x,
    abc,
):
    model.eval()

    total = 0.0
    n = 0
    alpha_sum = 0.0

    for i in range(
        0,
        len(
            x
        ),
        GATE_BATCH,
    ):
        xt = torch.from_numpy(
            x[
                i:
                i+GATE_BATCH
            ]
        ).to(
            DEVICE
        )

        at = torch.from_numpy(
            abc[
                i:
                i+GATE_BATCH
            ]
        ).to(
            DEVICE
        )

        alpha = model(
            xt
        )

        each = (
            at[
                :,
                0
            ]
            + 2.0
            * alpha
            * at[
                :,
                1
            ]
            + alpha
            * alpha
            * at[
                :,
                2
            ]
        )

        total += float(
            each.sum()
        )

        n += len(
            alpha
        )

        alpha_sum += float(
            alpha.sum()
        )

        del (
            xt,
            at,
            alpha,
            each,
        )

    return (
        total
        / n,
        alpha_sum
        / n,
    )


def train_crossfit_gate(
    name,
    horizon,
    oof_x,
    oof_abc,
    val_x,
    val_abc,
):
    path = gate_checkpoint_path(
        name,
        horizon,
    )

    if (
        path.exists()
        and RESUME
        and not FORCE
    ):
        ckpt = load_torch(
            path
        )

        model = CrossFitAdaptiveGate().to(
            DEVICE
        )

        model.load_state_dict(
            ckpt[
                "StateDict"
            ]
        )

        model.eval()

        print(
            "Loaded gate:",
            path.name,
        )

        return (
            model,
            ckpt,
        )

    median, iqr = fit_feature_scaler(
        oof_x
    )

    train_x = scale_features(
        oof_x,
        median,
        iqr,
    )

    valid_x = scale_features(
        val_x,
        median,
        iqr,
    )

    seed = (
        CROSSFIT_SEED
        + horizon
        * 3000
        + sum(
            map(
                ord,
                name,
            )
        )
    )

    set_seed(
        seed
    )

    model = CrossFitAdaptiveGate().to(
        DEVICE
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=GATE_LR,
        weight_decay=GATE_WD,
    )

    rng = np.random.default_rng(
        seed
        + 1
    )

    best = float(
        "inf"
    )

    best_epoch = -1
    wait = 0
    history = []

    for epoch in range(
        1,
        GATE_MAX_EPOCHS
        + 1,
    ):
        train_mse = train_gate_epoch(
            model,
            optimizer,
            train_x,
            oof_abc,
            rng,
        )

        val_mse, mean_alpha = evaluate_gate(
            model,
            valid_x,
            val_abc,
        )

        history.append({
            "Epoch":
                epoch,
            "OOFTrainMSE":
                train_mse,
            "ValMSE":
                val_mse,
            "ValMeanAlpha":
                mean_alpha,
        })

        if (
            val_mse
            < best
            - 1e-10
        ):
            best = val_mse
            best_epoch = epoch
            wait = 0
        else:
            wait += 1

        print(
            f"Gate {name:11s} H={horizon:3d} "
            f"ep={epoch:02d} "
            f"OOF={train_mse:.6f} "
            f"val={val_mse:.6f} "
            f"alpha={mean_alpha:.3f} "
            f"best={best:.6f}@{best_epoch}"
        )

        if (
            wait
            >= GATE_PATIENCE
        ):
            break

    # Reinitialize and fit only on OOF for the
    # validation-selected number of epochs.
    set_seed(
        seed
    )

    final = CrossFitAdaptiveGate().to(
        DEVICE
    )

    final_opt = torch.optim.AdamW(
        final.parameters(),
        lr=GATE_LR,
        weight_decay=GATE_WD,
    )

    final_rng = np.random.default_rng(
        seed
        + 2
    )

    for _ in range(
        best_epoch
    ):
        train_gate_epoch(
            final,
            final_opt,
            train_x,
            oof_abc,
            final_rng,
        )

    final.eval()

    ckpt = {
        "BestEpoch":
            best_epoch,
        "BestValMSE":
            best,
        "FeatureMedian":
            median,
        "FeatureIQR":
            iqr,
        "StateDict": {
            k:
                v.detach()
                .cpu()
                .clone()
            for k, v
            in final.state_dict().items()
        },
    }

    torch.save(
        ckpt,
        path,
    )

    pd.DataFrame(
        history
    ).to_csv(
        DIRS[
            "history"
        ]
        / (
            f"{name}_H{horizon}_"
            "gate_history.csv"
        ),
        index=False,
    )

    return (
        final,
        ckpt,
    )


def mse_scalar(
    abc,
    alpha,
):
    A = abc.astype(
        np.float64
    )

    x = float(
        alpha
    )

    return float(
        np.mean(
            A[
                :,
                0
            ]
            + 2.0
            * x
            * A[
                :,
                1
            ]
            + x
            * x
            * A[
                :,
                2
            ]
        )
    )


def choose_scalar(
    abc,
):
    rows = []

    best_alpha = None
    best_mse = float(
        "inf"
    )

    for alpha in ALPHA_GRID:
        mse = mse_scalar(
            abc,
            alpha,
        )

        rows.append({
            "Alpha":
                float(
                    alpha
                ),
            "MSE":
                mse,
        })

        if (
            mse
            < best_mse
            - 1e-10
        ):
            best_mse = mse
            best_alpha = float(
                alpha
            )

    return (
        best_alpha,
        pd.DataFrame(
            rows
        ),
    )


@torch.no_grad()
def gate_alpha(
    model,
    ckpt,
    x,
):
    sx = scale_features(
        x,
        ckpt[
            "FeatureMedian"
        ],
        ckpt[
            "FeatureIQR"
        ],
    )

    outputs = []

    for i in range(
        0,
        len(
            sx
        ),
        GATE_BATCH,
    ):
        t = torch.from_numpy(
            sx[
                i:
                i+GATE_BATCH
            ]
        ).to(
            DEVICE
        )

        outputs.append(
            model(
                t
            ).cpu()
            .numpy()
        )

        del t

    return np.concatenate(
        outputs
    ).astype(
        np.float32
    )


def mse_pair(
    abc,
    alpha,
):
    A = abc.astype(
        np.float64
    )

    x = np.asarray(
        alpha,
        dtype=np.float64,
    )

    return float(
        np.mean(
            A[
                :,
                0
            ]
            + 2.0
            * x
            * A[
                :,
                1
            ]
            + x
            * x
            * A[
                :,
                2
            ]
        )
    )


def choose_lambda(
    abc,
    gate_alpha_values,
    scalar_alpha,
):
    rows = []

    best_lambda = None
    best_mse = float(
        "inf"
    )

    for lmb in LAMBDA_GRID:
        alpha = (
            (
                1.0
                - float(
                    lmb
                )
            )
            * scalar_alpha
            + float(
                lmb
            )
            * gate_alpha_values
        )

        mse = mse_pair(
            abc,
            alpha,
        )

        rows.append({
            "Lambda":
                float(
                    lmb
                ),
            "MSE":
                mse,
            "MeanAlpha":
                float(
                    alpha.mean()
                ),
        })

        if (
            mse
            < best_mse
            - 1e-10
        ):
            best_mse = mse
            best_lambda = float(
                lmb
            )

    return (
        best_lambda,
        pd.DataFrame(
            rows
        ),
    )


In [ ]:

def empty_stat():
    return {
        "sse":
            0.0,
        "sae":
            0.0,
        "n":
            0,
    }


def update_stat(
    stat,
    pred,
    true,
):
    e = (
        pred
        - true
    )

    stat[
        "sse"
    ] += float(
        (
            e
            * e
        ).sum()
    )

    stat[
        "sae"
    ] += float(
        e.abs().sum()
    )

    stat[
        "n"
    ] += e.numel()


def finish_stat(
    stat,
):
    return (
        stat[
            "sse"
        ]
        / stat[
            "n"
        ],
        stat[
            "sae"
        ]
        / stat[
            "n"
        ],
    )


def oracle_alpha_and_prediction(
    direct,
    retrieval,
    true,
):
    e = (
        direct
        - true
    )

    delta = (
        retrieval
        - direct
    )

    alpha = torch.clamp(
        -(
            e
            * delta
        ).sum(
            dim=1
        )
        / (
            (
                delta
                * delta
            ).sum(
                dim=1
            )
            + 1e-8
        ),
        0.0,
        1.0,
    )

    pred = (
        direct
        + alpha[
            :,
            None
        ]
        * delta
    )

    return (
        alpha,
        pred,
    )


@torch.no_grad()
def test_evaluate(
    data,
    horizon,
    direct_model,
    retriever,
    memory_gpu_obj,
    gate,
    gate_ckpt,
    scalar_alpha,
    shrink_lambda,
):
    name = data[
        "name"
    ]

    C = data[
        "n_channels"
    ]

    direct_block = DIRECT_ANCHOR_BLOCK[
        name
    ]

    ret_block = retrieval_anchor_batch(
        name
    )

    anchors = eval_anchors(
        data[
            "val_end"
        ],
        data[
            "test_end"
        ],
        horizon,
        stride=1,
    )

    keys = [
        "Direct",
        "Retrieval",
        "Scalar",
        "RawAdaptive",
        "ShrinkAdaptive",
        "Oracle",
    ]

    stats = {
        key:
            empty_stat()
        for key in keys
    }

    anchor_mse = {
        key:
            []
        for key in keys
    }

    channel_sse_direct = np.zeros(
        C,
        dtype=np.float64,
    )

    channel_sse_shrink = np.zeros(
        C,
        dtype=np.float64,
    )

    channel_count = np.zeros(
        C,
        dtype=np.int64,
    )

    raw_alpha_sum = 0.0
    shrink_alpha_sum = 0.0
    oracle_alpha_sum = 0.0
    oracle_positive = 0
    n_pairs = 0

    processed = 0

    for outer in range(
        0,
        len(
            anchors
        ),
        direct_block,
    ):
        a_big = anchors[
            outer:
            outer+direct_block
        ]

        direct_big, true_big = (
            direct_residual_block(
                direct_model,
                data[
                    "z"
                ],
                data[
                    "marks"
                ],
                a_big,
                horizon,
            )
        )

        direct_big = direct_big.permute(
            0,
            2,
            1,
        ).contiguous()

        true_big = true_big.permute(
            0,
            2,
            1,
        ).contiguous()

        # anchor-level MSE accumulators within this direct block
        block_anchor_sums = {
            key:
                torch.zeros(
                    len(
                        a_big
                    ),
                    device=DEVICE,
                    dtype=torch.float64,
                )
            for key in keys
        }

        for inner in range(
            0,
            len(
                a_big
            ),
            ret_block,
        ):
            a = a_big[
                inner:
                inner+ret_block
            ]

            A = len(
                a
            )

            pair_anchor = np.repeat(
                a,
                C,
            )

            pair_channel = np.tile(
                np.arange(
                    C,
                    dtype=np.int64,
                ),
                A,
            )

            r = retrieve(
                retriever,
                memory_gpu_obj,
                data[
                    "z"
                ],
                pair_anchor,
                pair_channel,
                horizon,
            )

            retrieval = r[
                "cand"
            ].mean(
                dim=1
            )

            direct = direct_big[
                inner:
                inner+A
            ].reshape(
                -1,
                horizon,
            )

            true = true_big[
                inner:
                inner+A
            ].reshape(
                -1,
                horizon,
            )

            scalar = (
                direct
                + scalar_alpha
                * (
                    retrieval
                    - direct
                )
            )

            features = gate_features(
                r,
                retrieval,
                direct,
            ).cpu().numpy().astype(
                np.float32
            )

            scaled = scale_features(
                features,
                gate_ckpt[
                    "FeatureMedian"
                ],
                gate_ckpt[
                    "FeatureIQR"
                ],
            )

            gate_alpha_values = gate(
                torch.from_numpy(
                    scaled
                ).to(
                    DEVICE
                )
            )

            shrink_alpha_values = (
                (
                    1.0
                    - shrink_lambda
                )
                * scalar_alpha
                + shrink_lambda
                * gate_alpha_values
            )

            raw_adaptive = (
                direct
                + gate_alpha_values[
                    :,
                    None
                ]
                * (
                    retrieval
                    - direct
                )
            )

            shrink_adaptive = (
                direct
                + shrink_alpha_values[
                    :,
                    None
                ]
                * (
                    retrieval
                    - direct
                )
            )

            (
                oracle_alpha,
                oracle,
            ) = oracle_alpha_and_prediction(
                direct,
                retrieval,
                true,
            )

            predictions = {
                "Direct":
                    direct,
                "Retrieval":
                    retrieval,
                "Scalar":
                    scalar,
                "RawAdaptive":
                    raw_adaptive,
                "ShrinkAdaptive":
                    shrink_adaptive,
                "Oracle":
                    oracle,
            }

            for key, pred in predictions.items():
                update_stat(
                    stats[
                        key
                    ],
                    pred,
                    true,
                )

                per_anchor = (
                    (
                        (
                            pred
                            - true
                        )
                        ** 2
                    )
                    .reshape(
                        A,
                        C,
                        horizon,
                    )
                    .mean(
                        dim=(
                            1,
                            2,
                        )
                    )
                    .double()
                )

                block_anchor_sums[
                    key
                ][
                    inner:
                    inner+A
                ] = per_anchor

            direct_e2 = (
                (
                    direct
                    - true
                )
                ** 2
            ).reshape(
                A,
                C,
                horizon,
            )

            shrink_e2 = (
                (
                    shrink_adaptive
                    - true
                )
                ** 2
            ).reshape(
                A,
                C,
                horizon,
            )

            channel_sse_direct += (
                direct_e2.sum(
                    dim=(
                        0,
                        2,
                    )
                ).cpu()
                .numpy()
            )

            channel_sse_shrink += (
                shrink_e2.sum(
                    dim=(
                        0,
                        2,
                    )
                ).cpu()
                .numpy()
            )

            channel_count += (
                A
                * horizon
            )

            raw_alpha_sum += float(
                gate_alpha_values.sum()
            )

            shrink_alpha_sum += float(
                shrink_alpha_values.sum()
            )

            oracle_alpha_sum += float(
                oracle_alpha.sum()
            )

            oracle_positive += int(
                (
                    oracle_alpha
                    > 0.01
                ).sum()
            )

            n_pairs += len(
                gate_alpha_values
            )

            del (
                r,
                retrieval,
                direct,
                true,
                scalar,
                features,
                scaled,
                gate_alpha_values,
                shrink_alpha_values,
                raw_adaptive,
                shrink_adaptive,
                oracle_alpha,
                oracle,
                predictions,
                direct_e2,
                shrink_e2,
            )

        for key in keys:
            anchor_mse[
                key
            ].extend(
                block_anchor_sums[
                    key
                ].cpu()
                .numpy()
                .astype(
                    np.float32
                )
                .tolist()
            )

        processed += len(
            a_big
        )

        if (
            processed
            == len(
                a_big
            )
            or processed
            % 500
            < len(
                a_big
            )
            or processed
            == len(
                anchors
            )
        ):
            print(
                f"  test anchors "
                f"{processed}/{len(anchors)}"
            )

        del (
            direct_big,
            true_big,
            block_anchor_sums,
        )

    return {
        "anchors":
            anchors,
        "metrics": {
            key:
                finish_stat(
                    value
                )
            for key, value
            in stats.items()
        },
        "anchor_mse": {
            key:
                np.asarray(
                    value,
                    dtype=np.float32,
                )
            for key, value
            in anchor_mse.items()
        },
        "channel_direct_mse":
            channel_sse_direct
            / channel_count,
        "channel_shrink_mse":
            channel_sse_shrink
            / channel_count,
        "raw_mean_alpha":
            raw_alpha_sum
            / n_pairs,
        "shrink_mean_alpha":
            shrink_alpha_sum
            / n_pairs,
        "oracle_mean_alpha":
            oracle_alpha_sum
            / n_pairs,
        "oracle_positive_fraction":
            oracle_positive
            / n_pairs,
    }


In [ ]:

def moving_block_bootstrap(
    difference,
    n_boot=5000,
    block=24,
    seed=222222,
):
    x = np.asarray(
        difference,
        dtype=np.float64,
    )

    n = len(
        x
    )

    L = min(
        block,
        n,
    )

    rng = np.random.default_rng(
        seed
    )

    n_blocks = int(
        np.ceil(
            n
            / L
        )
    )

    max_start = max(
        1,
        n
        - L
        + 1,
    )

    boot = np.empty(
        n_boot,
        dtype=np.float64,
    )

    for b in range(
        n_boot
    ):
        starts = rng.integers(
            0,
            max_start,
            size=n_blocks,
        )

        sample = np.concatenate(
            [
                x[
                    s:
                    s+L
                ]
                for s in starts
            ]
        )[
            :n
        ]

        boot[
            b
        ] = sample.mean()

    return {
        "MeanImprovement":
            float(
                x.mean()
            ),
        "CI_Low":
            float(
                np.quantile(
                    boot,
                    0.025,
                )
            ),
        "CI_High":
            float(
                np.quantile(
                    boot,
                    0.975,
                )
            ),
    }


In [ ]:

SUMMARY_PATH = (
    ROOT
    / "summary.csv"
)

BOOTSTRAP_PATH = (
    ROOT
    / "bootstrap.csv"
)

FOLD_PATH = (
    ROOT
    / "folds.csv"
)

existing = (
    pd.read_csv(
        SUMMARY_PATH
    )
    if (
        RESUME
        and SUMMARY_PATH.exists()
    )
    else pd.DataFrame()
)

summary_rows = (
    existing.to_dict(
        "records"
    )
    if len(
        existing
    )
    else []
)

bootstrap_rows = (
    pd.read_csv(
        BOOTSTRAP_PATH
    ).to_dict(
        "records"
    )
    if (
        RESUME
        and BOOTSTRAP_PATH.exists()
    )
    else []
)

fold_rows = (
    pd.read_csv(
        FOLD_PATH
    ).to_dict(
        "records"
    )
    if (
        RESUME
        and FOLD_PATH.exists()
    )
    else []
)


def already_done(
    name,
    horizon,
):
    if not len(
        existing
    ):
        return False

    return bool(
        (
            (
                existing[
                    "Dataset"
                ]
                == name
            )
            & (
                existing[
                    "Horizon"
                ]
                == horizon
            )
        ).any()
    )


for name, horizon in TASKS:
    if already_done(
        name,
        horizon,
    ):
        print(
            f"SKIP completed: "
            f"{name} H={horizon}"
        )
        continue

    start_time = time.time()

    data = DATA[
        name
    ]

    C = data[
        "n_channels"
    ]

    print(
        "\n"
        + "#"
        * 150
    )

    print(
        f"{name} | H={horizon} | "
        "iTransformer + FROZEN HISTORICAL MEMORY"
    )

    print(
        "#"
        * 150
    )

    # --------------------------------------------------------
    # 1. Frozen strong full direct.
    # --------------------------------------------------------
    direct_model, direct_ckpt = load_exp21_direct(
        name,
        horizon,
    )

    ref = exp21_reference(
        name,
        horizon,
    )

    direct_epochs = int(
        direct_ckpt[
            "BestEpoch"
        ]
    )

    print(
        f"Experiment 21 direct: "
        f"MSE={float(ref['Test_MSE']):.6f}, "
        f"MAE={float(ref['Test_MAE']):.6f}, "
        f"best_epoch={direct_epochs}"
    )

    # --------------------------------------------------------
    # 2. Frozen full retriever.
    # --------------------------------------------------------
    retriever, retriever_ckpt = load_frozen_retriever(
        full_retriever_ckpt_path(
            name,
            horizon,
        )
    )

    # --------------------------------------------------------
    # 3. OOF cross-fitting.
    # --------------------------------------------------------
    oof_parts = []

    for fold, (
        p0,
        p1,
    ) in enumerate(
        FOLDS,
        start=1,
    ):
        part = build_oof_fold(
            data,
            horizon,
            fold,
            p0,
            p1,
            direct_epochs,
        )

        oof_parts.append(
            part
        )

        fold_rows = [
            r
            for r in fold_rows
            if not (
                r.get(
                    "Dataset"
                )
                == name
                and int(
                    r.get(
                        "Horizon",
                        -1,
                    )
                )
                == horizon
                and int(
                    r.get(
                        "Fold",
                        -1,
                    )
                )
                == fold
            )
        ]

        fold_rows.append({
            "Dataset":
                name,
            "Horizon":
                horizon,
            "Fold":
                fold,
            "PrefixFrac":
                p0,
            "OOFEndFrac":
                p1,
            "Pairs":
                len(
                    part[
                        "feature"
                    ]
                ),
            "Anchors":
                len(
                    np.unique(
                        part[
                            "anchor"
                        ]
                    )
                ),
            "DirectFixedEpochs":
                direct_epochs,
            "RetrieverCheckpoint":
                str(
                    fold_retriever_ckpt_path(
                        name,
                        horizon,
                        fold,
                    )
                ),
        })

        pd.DataFrame(
            fold_rows
        ).to_csv(
            FOLD_PATH,
            index=False,
        )

    oof_x = np.concatenate(
        [
            p[
                "feature"
            ]
            for p in oof_parts
        ],
        axis=0,
    )

    oof_abc = np.concatenate(
        [
            p[
                "abc"
            ]
            for p in oof_parts
        ],
        axis=0,
    )

    print(
        "Total OOF pairs:",
        len(
            oof_x
        ),
    )

    # --------------------------------------------------------
    # 4. Validation.
    # --------------------------------------------------------
    val_data = build_validation_cache(
        data,
        horizon,
        direct_model,
        retriever,
    )

    gate, gate_ckpt = train_crossfit_gate(
        name,
        horizon,
        oof_x,
        oof_abc,
        val_data[
            "feature"
        ],
        val_data[
            "abc"
        ],
    )

    scalar_alpha, scalar_curve = choose_scalar(
        val_data[
            "abc"
        ]
    )

    raw_val_alpha = gate_alpha(
        gate,
        gate_ckpt,
        val_data[
            "feature"
        ],
    )

    shrink_lambda, lambda_curve = choose_lambda(
        val_data[
            "abc"
        ],
        raw_val_alpha,
        scalar_alpha,
    )

    scalar_curve.to_csv(
        DIRS[
            "calibration"
        ]
        / (
            f"{name}_H{horizon}_"
            "scalar_alpha.csv"
        ),
        index=False,
    )

    lambda_curve.to_csv(
        DIRS[
            "calibration"
        ]
        / (
            f"{name}_H{horizon}_"
            "shrink_lambda.csv"
        ),
        index=False,
    )

    print(
        f"Validation calibration | "
        f"alpha0={scalar_alpha:.2f} | "
        f"lambda={shrink_lambda:.2f} | "
        f"gateEpoch={gate_ckpt['BestEpoch']}"
    )

    del (
        oof_x,
        oof_abc,
        oof_parts,
        raw_val_alpha,
        val_data,
    )

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    # --------------------------------------------------------
    # 5. Static test retrieval memory: train + validation only.
    # --------------------------------------------------------
    test_memory = build_memory(
        data[
            "z"
        ],
        C,
        data[
            "val_end"
        ],
        horizon,
    )

    test_memory_gpu = memory_gpu_cached(
        name,
        horizon,
        "test_trainval_memory",
        retriever,
        test_memory,
        C,
    )

    test = test_evaluate(
        data,
        horizon,
        direct_model,
        retriever,
        test_memory_gpu,
        gate,
        gate_ckpt,
        scalar_alpha,
        shrink_lambda,
    )

    metrics = test[
        "metrics"
    ]

    (
        direct_mse,
        direct_mae,
    ) = metrics[
        "Direct"
    ]

    (
        retrieval_mse,
        retrieval_mae,
    ) = metrics[
        "Retrieval"
    ]

    (
        scalar_mse,
        scalar_mae,
    ) = metrics[
        "Scalar"
    ]

    (
        raw_mse,
        raw_mae,
    ) = metrics[
        "RawAdaptive"
    ]

    (
        shrink_mse,
        shrink_mae,
    ) = metrics[
        "ShrinkAdaptive"
    ]

    (
        oracle_mse,
        oracle_mae,
    ) = metrics[
        "Oracle"
    ]

    exp21_mse = float(
        ref[
            "Test_MSE"
        ]
    )

    exp21_mae = float(
        ref[
            "Test_MAE"
        ]
    )

    parity_diff = abs(
        direct_mse
        - exp21_mse
    )

    if (
        parity_diff
        >= DIRECT_PARITY_TOL
    ):
        raise RuntimeError(
            f"Final direct parity failed for "
            f"{name} H={horizon}: "
            f"{parity_diff}"
        )

    # --------------------------------------------------------
    # 6. Paired bootstrap.
    # --------------------------------------------------------
    bootstrap_rows = [
        r
        for r in bootstrap_rows
        if not (
            r.get(
                "Dataset"
            )
            == name
            and int(
                r.get(
                    "Horizon",
                    -1,
                )
            )
            == horizon
        )
    ]

    b_direct = moving_block_bootstrap(
        test[
            "anchor_mse"
        ][
            "Direct"
        ]
        - test[
            "anchor_mse"
        ][
            "ShrinkAdaptive"
        ],
        seed=
            222222
            + horizon
            + sum(
                map(
                    ord,
                    name,
                )
            ),
    )

    bootstrap_rows.append({
        "Dataset":
            name,
        "Horizon":
            horizon,
        "Comparison":
            "Direct-ShrinkAdaptive",
        **b_direct,
        "SignificantPositive":
            b_direct[
                "CI_Low"
            ]
            > 0.0,
    })

    b_scalar = moving_block_bootstrap(
        test[
            "anchor_mse"
        ][
            "Scalar"
        ]
        - test[
            "anchor_mse"
        ][
            "ShrinkAdaptive"
        ],
        seed=
            333333
            + horizon
            + sum(
                map(
                    ord,
                    name,
                )
            ),
    )

    bootstrap_rows.append({
        "Dataset":
            name,
        "Horizon":
            horizon,
        "Comparison":
            "Scalar-ShrinkAdaptive",
        **b_scalar,
        "SignificantPositive":
            b_scalar[
                "CI_Low"
            ]
            > 0.0,
    })

    pd.DataFrame(
        bootstrap_rows
    ).to_csv(
        BOOTSTRAP_PATH,
        index=False,
    )

    # --------------------------------------------------------
    # 7. Per-channel diagnostic.
    # --------------------------------------------------------
    channel_df = pd.DataFrame({
        "ChannelIndex":
            np.arange(
                C
            ),
        "ChannelName":
            data[
                "columns"
            ],
        "Direct_MSE":
            test[
                "channel_direct_mse"
            ],
        "ShrinkAdaptive_MSE":
            test[
                "channel_shrink_mse"
            ],
    })

    channel_df[
        "Improvement"
    ] = (
        channel_df[
            "Direct_MSE"
        ]
        - channel_df[
            "ShrinkAdaptive_MSE"
        ]
    )

    channel_df[
        "Improvement_pct"
    ] = (
        100.0
        * channel_df[
            "Improvement"
        ]
        / channel_df[
            "Direct_MSE"
        ]
    )

    channel_df.to_csv(
        DIRS[
            "channel"
        ]
        / (
            f"{name}_H{horizon}_"
            "channel_mse.csv"
        ),
        index=False,
    )

    improved_channel_fraction = float(
        (
            channel_df[
                "Improvement"
            ]
            > 0.0
        ).mean()
    )

    # --------------------------------------------------------
    # 8. Save anchor-level paired losses.
    # --------------------------------------------------------
    np.savez_compressed(
        DIRS[
            "paired"
        ]
        / (
            f"{name}_H{horizon}_"
            "anchor_mse.npz"
        ),
        TestAnchors=
            test[
                "anchors"
            ],
        Direct=
            test[
                "anchor_mse"
            ][
                "Direct"
            ],
        Retrieval=
            test[
                "anchor_mse"
            ][
                "Retrieval"
            ],
        Scalar=
            test[
                "anchor_mse"
            ][
                "Scalar"
            ],
        RawAdaptive=
            test[
                "anchor_mse"
            ][
                "RawAdaptive"
            ],
        ShrinkAdaptive=
            test[
                "anchor_mse"
            ][
                "ShrinkAdaptive"
            ],
        Oracle=
            test[
                "anchor_mse"
            ][
                "Oracle"
            ],
    )

    # --------------------------------------------------------
    # 9. Final summary.
    # --------------------------------------------------------
    row = {
        "Dataset":
            name,
        "Horizon":
            horizon,
        "Channels":
            C,
        "iTransformer_MSE":
            direct_mse,
        "iTransformer_MAE":
            direct_mae,
        "Experiment21Reference_MSE":
            exp21_mse,
        "Experiment21Reference_MAE":
            exp21_mae,
        "DirectParityAbsDiff":
            parity_diff,
        "Retrieval_MSE":
            retrieval_mse,
        "Retrieval_MAE":
            retrieval_mae,
        "ScalarAlpha":
            scalar_alpha,
        "Scalar_MSE":
            scalar_mse,
        "Scalar_MAE":
            scalar_mae,
        "RawAdaptive_MSE":
            raw_mse,
        "RawAdaptive_MAE":
            raw_mae,
        "ShrinkLambda":
            shrink_lambda,
        "ShrinkAdaptive_MSE":
            shrink_mse,
        "ShrinkAdaptive_MAE":
            shrink_mae,
        "Oracle_MSE":
            oracle_mse,
        "Oracle_MAE":
            oracle_mae,
        "RawMeanAlpha":
            test[
                "raw_mean_alpha"
            ],
        "ShrinkMeanAlpha":
            test[
                "shrink_mean_alpha"
            ],
        "OracleMeanAlpha":
            test[
                "oracle_mean_alpha"
            ],
        "OraclePositiveFraction":
            test[
                "oracle_positive_fraction"
            ],
        "ImprovedChannelFraction":
            improved_channel_fraction,
        "ScalarGainVsDirect_pct":
            (
                100.0
                * (
                    direct_mse
                    - scalar_mse
                )
                / direct_mse
            ),
        "ShrinkGainVsDirect_pct":
            (
                100.0
                * (
                    direct_mse
                    - shrink_mse
                )
                / direct_mse
            ),
        "ShrinkGainVsScalar_pct":
            (
                100.0
                * (
                    scalar_mse
                    - shrink_mse
                )
                / scalar_mse
            ),
        "OracleHeadroomFromDirect_pct":
            (
                100.0
                * (
                    direct_mse
                    - oracle_mse
                )
                / direct_mse
            ),
        "OracleHeadroomFromShrink_pct":
            (
                100.0
                * (
                    shrink_mse
                    - oracle_mse
                )
                / shrink_mse
            ),
        "DirectBestEpoch":
            direct_epochs,
        "FrozenRetrieverBestEpoch":
            retriever_ckpt.get(
                "BestEpoch",
                np.nan,
            ),
        "GateBestEpoch":
            int(
                gate_ckpt[
                    "BestEpoch"
                ]
            ),
        "TestMemoryPerChannel":
            int(
                test_memory[
                    "M"
                ]
            ),
        "TestWindows":
            len(
                test[
                    "anchors"
                ]
            ),
        "RuntimeMinutes":
            (
                time.time()
                - start_time
            )
            / 60.0,
    }

    summary_rows = [
        r
        for r in summary_rows
        if not (
            r.get(
                "Dataset"
            )
            == name
            and int(
                r.get(
                    "Horizon",
                    -1,
                )
            )
            == horizon
        )
    ]

    summary_rows.append(
        row
    )

    pd.DataFrame(
        summary_rows
    ).to_csv(
        SUMMARY_PATH,
        index=False,
    )

    print(
        "\nFINAL CONDITION RESULT"
    )

    display(
        pd.DataFrame([
            row
        ])[
            [
                "Dataset",
                "Horizon",
                "iTransformer_MSE",
                "ShrinkAdaptive_MSE",
                "ShrinkGainVsDirect_pct",
                "iTransformer_MAE",
                "ShrinkAdaptive_MAE",
                "Retrieval_MSE",
                "Scalar_MSE",
                "RawAdaptive_MSE",
                "Oracle_MSE",
                "ScalarAlpha",
                "ShrinkLambda",
                "ShrinkMeanAlpha",
                "ImprovedChannelFraction",
                "OracleHeadroomFromDirect_pct",
                "RuntimeMinutes",
            ]
        ]
    )

    del (
        direct_model,
        direct_ckpt,
        retriever,
        retriever_ckpt,
        gate,
        gate_ckpt,
        test_memory,
        test_memory_gpu,
        test,
        channel_df,
    )

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


summary_df = (
    pd.DataFrame(
        summary_rows
    )
    .sort_values(
        [
            "Dataset",
            "Horizon",
        ]
    )
    .reset_index(
        drop=True
    )
)

display(
    summary_df
)


In [ ]:

if not len(
    summary_df
):
    raise RuntimeError(
        "No completed Experiment 22 result."
    )

compact = summary_df[
    [
        "Dataset",
        "Horizon",
        "iTransformer_MSE",
        "ShrinkAdaptive_MSE",
        "ShrinkGainVsDirect_pct",
        "iTransformer_MAE",
        "ShrinkAdaptive_MAE",
        "ScalarAlpha",
        "ShrinkLambda",
        "ShrinkMeanAlpha",
        "ImprovedChannelFraction",
        "Oracle_MSE",
        "OracleHeadroomFromDirect_pct",
        "DirectParityAbsDiff",
    ]
].copy()

compact[
    "Winner"
] = np.where(
    compact[
        "ShrinkAdaptive_MSE"
    ]
    < compact[
        "iTransformer_MSE"
    ],
    "Ours",
    "Direct",
)

display(
    compact
)

compact.to_csv(
    ROOT
    / "compact_results.csv",
    index=False,
)

if BOOTSTRAP_PATH.exists():
    boot_df = pd.read_csv(
        BOOTSTRAP_PATH
    )

    direct_boot = boot_df[
        boot_df[
            "Comparison"
        ]
        == "Direct-ShrinkAdaptive"
    ].copy()

    direct_boot[
        "Significance"
    ] = np.select(
        [
            direct_boot[
                "CI_Low"
            ]
            > 0.0,
            direct_boot[
                "CI_High"
            ]
            < 0.0,
        ],
        [
            "Ours significantly better",
            "Direct significantly better",
        ],
        default=
            "Not significant",
    )

    display(
        direct_boot.sort_values(
            [
                "Dataset",
                "Horizon",
            ]
        )
    )


In [ ]:

if len(
    summary_df
):
    runtime = (
        summary_df
        .groupby(
            "Dataset",
            as_index=False,
        )
        .agg(
            Conditions=(
                "Horizon",
                "size",
            ),
            TotalRuntimeMinutes=(
                "RuntimeMinutes",
                "sum",
            ),
            MeanRuntimeMinutes=(
                "RuntimeMinutes",
                "mean",
            ),
        )
    )

    runtime[
        "TotalRuntimeHours"
    ] = (
        runtime[
            "TotalRuntimeMinutes"
        ]
        / 60.0
    )

    display(
        runtime
    )

    print(
        "Recorded total runtime (hours):",
        summary_df[
            "RuntimeMinutes"
        ].sum()
        / 60.0,
    )


In [ ]:

print(
    "Experiment root:",
    ROOT,
)

for p in sorted(
    ROOT.rglob(
        "*"
    )
):
    if p.is_file():
        print(
            p.relative_to(
                ROOT
            )
        )
